# Effet de alpha : impact de la collaboration

Ce notebook lit les résultats d'une **campagne** (par défaut `alpha_sweep`, voir `configs/campaigns/`) pour une
méthode (`METHOD`) et un groupe (dataset, K), avec les agents indépendants (alpha = 0) et le grand modèle seul
(`results/reference/`). Il affiche :

1. la configuration ;
2. le tableau de synthèse par alpha (médiane [quantile 5 % ; quantile 95 %] sur les graines) ;
3. l'accuracy en fonction de alpha (consensus, agent médian, meilleur agent), grand modèle en référence ;
4. le gain de la collaboration par rapport aux agents indépendants (alpha = 0) ;
5. la diversité des agents (désaccord) en fonction de alpha ;
6. les courbes d'apprentissage pour chaque alpha ;
7. la calibration (NLL, ECE) en fonction de alpha.

Régler les valeurs de la première cellule puis *Run All*. Les figures vont dans
`results/<campagne>/<dataset>/<arch>_<K>agents/figures/notebooks/`.

In [ ]:
import json
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from IPython.display import Markdown, display

# --- À régler -------------------------------------------------------------
CODE_DIR = ".."                # racine du dépôt (contient collaborative_learning/ et configs/)
RESULTS_ROOT = "../results"    # dossier des résultats ("../results_quick_test" pour la campagne quick_test)
CAMPAIGN = "alpha_sweep"        # nom dans configs/campaigns/, ou chemin d'un fichier de campagne
DATASET = None                 # None = le premier dataset de la campagne, sinon ex. "cifar10"
K = None                       # None = le premier K de la campagne
METHOD = None                  # None = la première méthode de la campagne, sinon ex. "dml"
FIXED = {}                     # autres valeurs de la grille à garder, ex. {"lr": 0.01}
# ---------------------------------------------------------------------------

sys.path.insert(0, os.path.abspath(CODE_DIR))
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
from experiments.campaign import Campaign, point_name
from collaborative_learning.evaluation.plots import (BIG_COLOR, campaign_groups, constant_keys, entry_name, method_color,
                                                     method_label, method_ramp)

campaign = Campaign(CAMPAIGN, {"out_dir": RESULTS_ROOT})
groups = {g: v for g, v in campaign_groups(campaign).items()
          if (DATASET is None or g[0] == DATASET) and (K is None or g[2] == K) and v["entries"]}
if not groups:
    raise FileNotFoundError(f"Aucun résultat fusionné pour la campagne {campaign.name} dans {os.path.abspath(RESULTS_ROOT)} "
                            f"(python -m experiments.campaign {campaign.name} --merge)")
(DATASET, ARCH, K), group = next(iter(groups.items()))
entries, reference, GROUP_DIR = group["entries"], group["reference"], group["dir"]
print(f"Campagne {campaign.name} : {DATASET}, {K} × {ARCH}  |  {len(entries)} dossier(s) de K agents  |  "
      f"grand modèle : {os.path.relpath(reference['dir'], RESULTS_ROOT) if reference else 'aucun'}")

METHOD = METHOD or campaign.methods[0]

def keep(e):
    return e["method"] in (METHOD, "independent") and all(
        k in e["point"] and float(e["point"][k]) == float(v) for k, v in FIXED.items())

selected = [e for e in entries if keep(e)]
runs = [r for e in selected for r in e["results"]["runs"]]
alphas = sorted({float(r["alpha"]) for r in runs})
if len(alphas) < 2:
    raise ValueError(f"{METHOD} : moins de deux valeurs de alpha terminées ({alphas}) : ce notebook demande un balayage "
                     f"de alpha (ex. CAMPAIGN = 'alpha_sweep') ; méthodes de la campagne : {sorted({e['method'] for e in entries})}")
config = next((e["config"] for e in selected if e["method"] == METHOD), selected[0]["config"])
results = {"train_time_s": sum(e["results"].get("train_time_s", 0.0) for e in selected),
           "agent_params": next((e["results"].get("agent_params") for e in selected), None)}
seeds = sorted({r["seed"] for r in runs})
by_alpha = {a: [r for r in runs if float(r["alpha"]) == a] for a in alphas}
big_runs = reference["results"]["runs"] if reference else []   # grand modèle seul
print(f"Méthode : {method_label(METHOD)}  |  alphas : {[f'{a:g}' for a in alphas]}  |  graines : {seeds}  |  runs : {len(runs)}")

In [ ]:
# --- Style commun des figures ----------------------------------------------
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, AXIS, SURFACE = "#e1e0d9", "#c3c2b7", "#fcfcfb"
CONS = method_color(METHOD)              # consensus : couleur de la méthode (la même que dans plot_results)
AGENT = method_ramp(METHOD, 3)[0]        # agent médian : nuance claire de la même couleur
GRAY = "#a9a79f"                         # meilleur agent / contexte ; grand modèle en noir pointillé
# Gamme de la méthode : un alpha plus grand = plus foncé
RAMP = method_ramp(METHOD, 7)

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "font.size": 10.5, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.titlepad": 10,
    "axes.labelcolor": INK2, "axes.edgecolor": AXIS, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "legend.frameon": False, "lines.linewidth": 2, "lines.solid_capstyle": "round",
})

FIG_DIR = os.path.join(GROUP_DIR, "figures", "notebooks", "_".join(
    ["effect_of_alpha", METHOD] + [f"{k}{v:g}" for k, v in FIXED.items()]))
os.makedirs(FIG_DIR, exist_ok=True)

def save(fig, name):
    fig.savefig(os.path.join(FIG_DIR, f"{name}.png"))

Q_LO, Q_HI = 0.05, 0.95

def stats(values, axis=None):
    """Médiane, quantile 5 % et quantile 95 % (sur les graines)."""
    v = np.asarray(values, dtype=float)
    return np.median(v, axis=axis), np.quantile(v, Q_LO, axis=axis), np.quantile(v, Q_HI, axis=axis)

def per_alpha(key, transform=lambda r, k: r[k]):
    """Pour chaque alpha : (médiane, q05, q95) de la métrique sur les graines (None si absente)."""
    out = []
    for a in alphas:
        vals = [transform(r, key) for r in by_alpha[a]]
        vals = [v for v in vals if v is not None]
        out.append(stats(vals) if vals else (np.nan, np.nan, np.nan))
    return np.array(out).T   # (3, n_alphas)

# alpha sur un axe à positions régulières (0 ; 0,5 ; 1 ; 10 ; 50 ; 100 sont trop inégaux pour un axe linéaire)
X = np.arange(len(alphas))
XLABELS = [f"{a:g}" for a in alphas]
def alpha_axis(ax):
    ax.set_xticks(X, XLABELS)
    ax.set_xlabel("alpha  (0 = agents indépendants, vote majoritaire)")
    ax.grid(axis="x", visible=False)

K = config["num_agents"]
TITLE = f"{K} × {ARCH} — {method_label(METHOD)}"

## 1. Configuration

In [ ]:
rows = {
    "Jeu de données": config["dataset"],
    "Agents": f"{K} × {config['arch']}  ({results['agent_params'] or 0:,} param. chacun)",
    "Campagne": f"{campaign.name} ({os.path.relpath(GROUP_DIR, RESULTS_ROOT)})",
    "Loss": f"α > 0 : {config['loss_type']} ({config['loss_form']}) ; α = 0 : cross-entropy seule (agents indépendants)",
    "Alphas": ", ".join(XLABELS),
    "Stratégie de consensus": f"majority pour α = 0, {config['strategy']} sinon (τ = {config['temperature']})",
    "Optimiseur": f"{config['optimizer']}, lr = {config['lr']}" + (f", {config['optimizer_kwargs']}" if config.get("optimizer_kwargs") else ""),
    "Entraînement": f"{config['epochs']} epochs (warmup {config['warmup_epochs']}), batch {config['batch_size']}, weight decay {config['weight_decay']}, augmentation : {config['augment']}",
    "Validation / sous-ensemble": f"val_size = {config['val_size']}, train_subset = {config['train_subset']}",
    "Graines": ", ".join(map(str, seeds)),
    "Matériel": f"{config.get('device', '?')}, précision {config['precision']}",
    "Durée d'entraînement (somme des tâches)": f"{results.get('train_time_s', float('nan')) / 60:.1f} min",
}
display(pd.DataFrame({"": list(rows.values())}, index=list(rows.keys()))
        .style.set_properties(**{"text-align": "left"})
        .set_table_styles([{"selector": "th", "props": [("text-align", "left"), ("font-weight", "600")]}]))

## 2. Tableau de synthèse

Médiane [quantile 5 % ; quantile 95 %] sur les graines. En gras : le meilleur alpha de chaque colonne. Avec peu de graines, les quantiles sont proches du minimum et du maximum.

In [ ]:
med_agent = lambda key: (lambda r, k: float(np.median(r[key])))
COLUMNS = {
    "Consensus (%)": ("accuracy_consensus", None, 100, "max"),
    "Moyenne simple (%)": ("accuracy_mean_ensemble", None, 100, "max"),
    "Agent médian (%)": ("accuracy_agent_median", None, 100, "max"),
    "Meilleur agent (%)": ("accuracy_agent_best", None, 100, "max"),
    "Désaccord (%)": ("disagreement", None, 100, None),
    "NLL consensus": ("nll_consensus", None, 1, "min"),
    "ECE consensus": ("ece_consensus", None, 1, "min"),
    "NLL agent médian": ("nll_agents", med_agent("nll_agents"), 1, "min"),
}
table, medians = {}, {}
for col, (key, tf, scale, _) in COLUMNS.items():
    m, lo, hi = per_alpha(key, tf) if tf else per_alpha(key)
    m, lo, hi = m * scale, lo * scale, hi * scale
    digits = 2 if scale == 100 else 4
    table[col] = [f"{a:.{digits}f} [{b:.{digits}f} ; {c:.{digits}f}]" if not np.isnan(a) else "–"
                  for a, b, c in zip(m, lo, hi)]
    medians[col] = m
index = [f"{a:g} ({by_alpha[a][0]['strategy']})" for a in alphas]
table = pd.DataFrame(table, index=pd.Index(index, name="alpha"))
medians = pd.DataFrame(medians, index=index)

def highlight(col):
    rule = COLUMNS[col.name][3]
    if rule is None or medians[col.name].isna().all():
        return [""] * len(col)
    best = medians[col.name].idxmax() if rule == "max" else medians[col.name].idxmin()
    return ["font-weight: 700" if i == best else "" for i in col.index]

display(table.style.apply(highlight)
        .set_properties(**{"text-align": "right", "font-variant-numeric": "tabular-nums"})
        .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}]))

## 3. Accuracy en fonction de alpha

Ligne : médiane sur les graines ; bande : quantiles 5 %–95 % ; points : chaque graine.

In [ ]:
series = [
    ("consensus", "accuracy_consensus", CONS, "-"),
    ("agent médian", "accuracy_agent_median", AGENT, "-"),
    ("meilleur agent", "accuracy_agent_best", GRAY, "--"),
]
fig, ax = plt.subplots(figsize=(8.5, 4.6))
for label, key, color, ls in series:
    m, lo, hi = per_alpha(key) * 100
    ax.fill_between(X, lo, hi, color=color, alpha=0.13, lw=0)
    ax.plot(X, m, color=color, ls=ls, marker="o", ms=7, mec=SURFACE, mew=1.5, label=label, zorder=3)
    if key != "accuracy_agent_best":
        for j, a in enumerate(alphas):
            vals = [r[key] * 100 for r in by_alpha[a]]
            ax.scatter(np.full(len(vals), X[j]), vals, s=12, color=color, alpha=0.45, lw=0, zorder=2)
    ax.annotate(f"{m[-1]:.1f}", (X[-1], m[-1]), xytext=(8, 0), textcoords="offset points",
                va="center", color=INK, fontsize=9)
if big_runs:   # grand modèle seul : ne dépend pas de alpha
    bm, blo, bhi = stats([r["accuracy"] * 100 for r in big_runs])
    ax.axhspan(blo, bhi, color=INK2, alpha=0.07, lw=0)
    ax.axhline(bm, color=INK, ls=":", lw=2, label="grand modèle seul")
alpha_axis(ax)
ax.set_xlim(-0.3, len(alphas) - 0.4)
ax.set_ylabel("Accuracy de test (%)")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
ax.set_title(f"Accuracy selon la force de collaboration — {TITLE}")
save(fig, "accuracy_vs_alpha"); plt.show()

## 4. Gain de la collaboration

À gauche : gain du consensus sur l'agent médian **dans le même run** (ce que le groupe apporte par rapport à un de ses membres). À droite : accuracy de l'agent médian par rapport à alpha = 0 (agents indépendants) **sur la même graine** — ce que la collaboration apporte à chaque agent.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

# Gain consensus - agent médian, par run
gain = [[(r["accuracy_consensus"] - r["accuracy_agent_median"]) * 100 for r in by_alpha[a]] for a in alphas]
m = np.array([stats(g) for g in gain]).T
ax = axes[0]
ax.axhline(0, color=AXIS, lw=1, zorder=1)
ax.bar(X, m[0], width=0.6, color=CONS, zorder=2)
ax.errorbar(X, m[0], yerr=[m[0] - m[1], m[2] - m[0]], fmt="none", ecolor=INK2, lw=1.2, capsize=3, zorder=3)
for x, v in zip(X, m[0]):
    ax.annotate(f"{v:+.1f}", (x, v), xytext=(0, 4 if v >= 0 else -12), textcoords="offset points",
                ha="center", fontsize=9, color=INK)
alpha_axis(ax)
ax.set_ylabel("points d'accuracy")
ax.set_title("Consensus − agent médian")

# Agent médian vs alpha = 0, même graine
ax = axes[1]
if 0.0 in by_alpha:
    base = {r["seed"]: r["accuracy_agent_median"] for r in by_alpha[0.0]}
    delta = [[(r["accuracy_agent_median"] - base[r["seed"]]) * 100 for r in by_alpha[a] if r["seed"] in base]
             for a in alphas]
    m = np.array([stats(d) if d else (np.nan,) * 3 for d in delta]).T
    ax.axhline(0, color=AXIS, lw=1, zorder=1)
    colors = [GRAY if a == 0 else AGENT for a in alphas]
    ax.bar(X, m[0], width=0.6, color=colors, zorder=2)
    ax.errorbar(X, m[0], yerr=[m[0] - m[1], m[2] - m[0]], fmt="none", ecolor=INK2, lw=1.2, capsize=3, zorder=3)
    for x, v in zip(X, m[0]):
        if not np.isnan(v):
            ax.annotate(f"{v:+.1f}", (x, v), xytext=(0, 4 if v >= 0 else -12), textcoords="offset points",
                        ha="center", fontsize=9, color=INK)
    ax.set_title("Agent médian : écart à alpha = 0")
else:
    ax.text(0.5, 0.5, "alpha = 0 absent du balayage", ha="center", va="center", transform=ax.transAxes, color=MUTED)
    ax.set_title("Agent médian : écart à alpha = 0")
alpha_axis(ax)
ax.set_ylabel("points d'accuracy")
fig.tight_layout()
save(fig, "collaboration_gain"); plt.show()

## 5. Diversité des agents

Pourcentage moyen d'exemples de test sur lesquels deux agents prédisent une classe différente. Une collaboration forte rapproche les agents ; s'ils deviennent identiques, le consensus n'apporte plus rien.

In [ ]:
m, lo, hi = per_alpha("disagreement") * 100
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.fill_between(X, lo, hi, color=CONS, alpha=0.13, lw=0)
ax.plot(X, m, color=CONS, marker="o", ms=7, mec=SURFACE, mew=1.5, zorder=3)
for x, v in zip(X, m):
    ax.annotate(f"{v:.1f}", (x, v), xytext=(0, 8), textcoords="offset points", ha="center", fontsize=9, color=INK)
alpha_axis(ax)
ax.set_ylim(bottom=0)
ax.set_ylabel("désaccord entre agents (%)")
ax.set_title("Diversité des agents selon alpha")
save(fig, "diversity_vs_alpha"); plt.show()

## 6. Courbes d'apprentissage

Accuracy du consensus (médiane sur les graines) pour chaque alpha ; plus le bleu est foncé, plus alpha est grand.

In [ ]:
keys = [("accuracy", "Train"), ("val_accuracy", "Validation")]
keys = [k for k in keys if all(k[0] in r["history"] for r in runs)]
fig, axes = plt.subplots(1, len(keys), figsize=(6 * len(keys), 4), sharey=True, squeeze=False)
ramp = method_ramp(METHOD, len(alphas))
for ax, (key, title) in zip(axes[0], keys):
    for a, color in zip(alphas, ramp):
        curves = np.array([r["history"][key] for r in by_alpha[a]]) * 100
        x = np.arange(1, curves.shape[1] + 1)
        ax.plot(x, np.median(curves, 0), color=color, label=f"α = {a:g}")
    ax.set_title(f"Accuracy du consensus — {title}")
    ax.set_xlabel("epoch")
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
axes[0][0].set_ylabel("Accuracy (%)")
axes[0][-1].legend(loc="upper left", bbox_to_anchor=(1.02, 1))
save(fig, "learning_curves_vs_alpha"); plt.show()

## 7. Calibration

NLL et ECE : plus bas = mieux. Pour alpha = 0 (vote majoritaire), le consensus ne donne pas de probabilités : seule la valeur de l'agent médian est affichée.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (metric, label) in zip(axes, [("nll", "NLL"), ("ece", "ECE")]):
    for name, key, tf, color in [
        ("consensus", f"{metric}_consensus", None, CONS),
        ("agent médian", f"{metric}_agents", lambda r, k: float(np.median(r[k])), AGENT),
    ]:
        m, lo, hi = per_alpha(key, tf) if tf else per_alpha(key)
        ok = ~np.isnan(m)
        ax.fill_between(X[ok], lo[ok], hi[ok], color=color, alpha=0.13, lw=0)
        ax.plot(X[ok], m[ok], color=color, marker="o", ms=7, mec=SURFACE, mew=1.5, label=name, zorder=3)
    alpha_axis(ax)
    ax.set_title(f"{label} selon alpha")
    ax.set_ylabel(label)
axes[1].legend(loc="upper left", bbox_to_anchor=(1.02, 1))
fig.tight_layout()
save(fig, "calibration_vs_alpha"); plt.show()
print(f"Figures enregistrées dans {os.path.abspath(FIG_DIR)}")